In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

print("--- PART 1: TITANIC HYPERPARAMETER TUNING (GridSearchCV) ---")

# 1. Load & Preprocess Data (Same as Assignment 3)
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_titanic = pd.read_csv(titanic_url)
df_titanic = df_titanic[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']].copy()
df_titanic['Age'] = df_titanic['Age'].fillna(df_titanic['Age'].median())
df_titanic['Sex'] = LabelEncoder().fit_transform(df_titanic['Sex'])

X_clf = df_titanic.drop('Survived', axis=1)
y_clf = df_titanic['Survived']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# 2. Initialize Base Model
xgb_clf_base = XGBClassifier(eval_metric='logloss', random_state=42)

# 3. Define the Parameter Grid
# GridSearchCV will try 3 x 3 x 3 = 27 different model combinations
param_grid_clf = {
    'max_depth': [3, 5, 7],             # How deep the trees can grow
    'learning_rate': [0.01, 0.1, 0.2],  # How fast the model learns
    'n_estimators': [50, 100, 200]      # Number of trees to build
}

# 4. Set up and run GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_clf_base, 
    param_grid=param_grid_clf, 
    cv=5,                 # 5-Fold Cross Validation
    scoring='accuracy', 
    n_jobs=-1,            # Uses all your computer's CPU cores to run faster
    verbose=1
)

print("Running Grid Search for Titanic...")
grid_search.fit(X_train_c, y_train_c)

# 5. Evaluate the Best Model
best_clf = grid_search.best_estimator_
y_pred_c = best_clf.predict(X_test_c)

print(f"\nBest Parameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")
print(f"Final Test Accuracy: {accuracy_score(y_test_c, y_pred_c):.4f}")
print("-" * 50)

--- PART 1: TITANIC HYPERPARAMETER TUNING (GridSearchCV) ---
Running Grid Search for Titanic...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

Best Parameters Found: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 50}
Best Cross-Validation Accuracy: 0.8343
Final Test Accuracy: 0.8324
--------------------------------------------------


In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

print("\n--- PART 2: HOUSE PRICE HYPERPARAMETER TUNING (RandomizedSearchCV) ---")

# 1. Load & Preprocess Data (Same as Assignment 3)
house_prices_url = "https://raw.githubusercontent.com/eddiexunyc/DATA605_FINAL/main/Resources/train.csv"
df_house = pd.read_csv(house_prices_url)

num_cols = df_house.select_dtypes(include=[np.number]).columns
df_house = df_house[num_cols].copy()
if 'Id' in df_house.columns:
    df_house = df_house.drop('Id', axis=1)

df_house = df_house.fillna(df_house.median())

X_reg = df_house.drop('SalePrice', axis=1)
y_reg = df_house['SalePrice']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 2. Initialize Base Model
xgb_reg_base = XGBRegressor(random_state=42)

# 3. Define the Parameter Distribution
# We give it a wider range of options, and it will pick randomly from them
param_dist_reg = {
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 500],
    'subsample': [0.7, 0.8, 0.9, 1.0],         # % of rows used per tree
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]   # % of columns used per tree
}

# 4. Set up and run RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_reg_base, 
    param_distributions=param_dist_reg, 
    n_iter=20,            # It will only try 20 random combinations out of the hundreds possible
    cv=5,                 # 5-Fold Cross Validation
    scoring='neg_root_mean_squared_error', 
    n_jobs=-1, 
    verbose=1,
    random_state=42       # Ensures reproducibility
)

print("Running Randomized Search for House Prices...")
random_search.fit(X_train_r, y_train_r)

# 5. Evaluate the Best Model
best_reg = random_search.best_estimator_
y_pred_r = best_reg.predict(X_test_r)

print(f"\nBest Parameters Found: {random_search.best_params_}")
# Note: We multiply by -1 because scikit-learn stores it as a negative number internally
print(f"Best Cross-Validation RMSE: {-random_search.best_score_:.2f}")
print(f"Final Test RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.2f}")
print(f"Final Test R-squared: {r2_score(y_test_r, y_pred_r):.4f}")


--- PART 2: HOUSE PRICE HYPERPARAMETER TUNING (RandomizedSearchCV) ---
Running Randomized Search for House Prices...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Parameters Found: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
Best Cross-Validation RMSE: 29227.91
Final Test RMSE: 26755.25
Final Test R-squared: 0.9067
